# 🫀 Notebook 1 — Dataset Loading and Exploration
### Project: Explainable Lightweight Edge-AI Framework for Early Cardiac Risk Prediction using Attention-Based Deep Learning on ECG Signals
**Week 1 | Task 1: Dataset Loading, Inspection, and Visualization**

---
**Datasets Used:**
- MIT-BIH Arrhythmia Dataset (Primary Training)
- PTB Diagnostic ECG Dataset (External Validation)

**Objectives:**
- Load ECG signals using WFDB library
- Inspect dataset structure and annotations
- Visualize raw ECG waveforms
- Generate dataset statistics
---

## 📦 Cell 1 — Import Required Libraries

In [ ]:
# ── Standard Libraries ──────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# ── Numerical & Data Libraries ───────────────────────────────────
import numpy as np
import pandas as pd

# ── ECG-Specific Libraries ────────────────────────────────────────
import wfdb                        # PhysioNet ECG reading
import neurokit2 as nk             # ECG signal processing

# ── Visualization Libraries ───────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Style Configuration ───────────────────────────────────────────
plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a2e'
plt.rcParams['axes.edgecolor']   = '#444'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['grid.color']       = '#333'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['grid.alpha']       = 0.5
sns.set_palette('husl')

print('✅ All libraries imported successfully!')
print(f'   NumPy     : {np.__version__}')
print(f'   Pandas    : {pd.__version__}')
print(f'   WFDB      : {wfdb.__version__}')
print(f'   NeuroKit2 : {nk.__version__}')

**Expected Output:**
```
✅ All libraries imported successfully!
   NumPy     : 1.x.x
   Pandas    : 2.x.x
   WFDB      : 4.x.x
   NeuroKit2 : 0.x.x
```

---
## 📁 Cell 2 — Define Dataset Paths
> **⚠️ IMPORTANT:** Update the paths below to match where you saved the datasets on your machine.

In [ ]:
# ── Dataset Root Paths ────────────────────────────────────────────
# Update these paths to match your local folder structure
MIT_BIH_PATH = './data/mit-bih-arrhythmia-database-1.0.0'   # MIT-BIH folder
PTB_PATH     = './data/ptb-diagnostic-ecg-database-1.0.0'    # PTB folder

# ── MIT-BIH: List all available record numbers ───────────────────
# Standard MIT-BIH record numbers (48 records total)
MIT_BIH_RECORDS = [
    '100','101','102','103','104','105','106','107',
    '108','109','111','112','113','114','115','116',
    '117','118','119','121','122','123','124','200',
    '201','202','203','205','207','208','209','210',
    '212','213','214','215','217','219','220','221',
    '222','223','228','230','231','232','233','234'
]

print(f'📂 MIT-BIH Path   : {MIT_BIH_PATH}')
print(f'📂 PTB Path       : {PTB_PATH}')
print(f'📋 MIT-BIH Records: {len(MIT_BIH_RECORDS)} records available')

# ── Verify folders exist ──────────────────────────────────────────
for name, path in [('MIT-BIH', MIT_BIH_PATH), ('PTB', PTB_PATH)]:
    status = '✅ Found' if os.path.exists(path) else '❌ NOT FOUND — check your path'
    print(f'   {name}: {status}')

---
## 📖 Cell 3 — Load a Single MIT-BIH ECG Record

In [ ]:
# ── Load Record 100 (first MIT-BIH record) ────────────────────────
SAMPLE_RECORD = '100'

record_path = os.path.join(MIT_BIH_PATH, SAMPLE_RECORD)

# Load signal + metadata
record      = wfdb.rdrecord(record_path)

# Load annotations (beat labels)
annotation  = wfdb.rdann(record_path, 'atr')

# ── Display Record Info ───────────────────────────────────────────
print('=' * 55)
print(f'  📄 Record         : {record.record_name}')
print(f'  📡 Channels       : {record.n_sig}  →  {record.sig_name}')
print(f'  🔢 Total Samples  : {record.sig_len}')
print(f'  ⚡ Sampling Rate  : {record.fs} Hz')
print(f'  ⏱️  Duration       : {record.sig_len / record.fs:.1f} seconds')
print(f'  📏 Signal Units   : {record.units}')
print(f'  💓 Total Beats    : {len(annotation.sample)}')
print('=' * 55)

# ── Extract signal array ──────────────────────────────────────────
ecg_signal   = record.p_signal        # shape: (samples, channels)
sampling_rate = record.fs

print(f'\n📐 Signal Shape   : {ecg_signal.shape}  (samples × channels)')
print(f'📊 Signal Range   : [{ecg_signal.min():.4f},  {ecg_signal.max():.4f}] mV')
print(f'📈 Signal Mean    : {ecg_signal.mean():.6f} mV')
print(f'📉 Signal Std Dev : {ecg_signal.std():.6f} mV')

---
## 🏷️ Cell 4 — Explore Annotation Labels

In [ ]:
# ── Extract beat symbols & positions ─────────────────────────────
beat_symbols  = annotation.symbol          # Beat type labels
beat_samples  = annotation.sample          # Sample positions of each beat

# ── Count each beat type ──────────────────────────────────────────
unique_labels, counts = np.unique(beat_symbols, return_counts=True)
label_counts = dict(zip(unique_labels, counts))

# ── AAMI beat type descriptions ──────────────────────────────────
BEAT_DESCRIPTIONS = {
    'N': 'Normal Beat',
    'L': 'Left Bundle Branch Block',
    'R': 'Right Bundle Branch Block',
    'A': 'Atrial Premature Beat',
    'a': 'Aberrated Atrial Premature Beat',
    'J': 'Nodal (Junctional) Premature Beat',
    'S': 'Supraventricular Premature Beat',
    'V': 'Premature Ventricular Contraction',
    'F': 'Fusion of Ventricular and Normal Beat',
    '[': 'Start of Ventricular Flutter/Fibrillation',
    '!': 'Ventricular Flutter Wave',
    ']': 'End of Ventricular Flutter/Fibrillation',
    'e': 'Atrial Escape Beat',
    'j': 'Nodal (Junctional) Escape Beat',
    'E': 'Ventricular Escape Beat',
    '/': 'Paced Beat',
    'f': 'Fusion of Paced and Normal Beat',
    'x': 'Non-Conducted P-wave (Block)',
    'Q': 'Unclassifiable Beat',
    '|': 'Isolated QRS-like Artifact'
}

print('\n🏷️  Beat Type Distribution — Record 100')
print('-' * 50)
print(f'  {"Symbol":<10} {"Count":<10} {"Description"}')
print('-' * 50)
for sym, cnt in sorted(label_counts.items(), key=lambda x: -x[1]):
    desc = BEAT_DESCRIPTIONS.get(sym, 'Other/Non-beat annotation')
    print(f'  {sym:<10} {cnt:<10} {desc}')
print('-' * 50)
print(f'  {"TOTAL":<10} {sum(counts)}')

---
## 📊 Cell 5 — Visualize Raw ECG Waveform (10 seconds)

In [ ]:
# ── Plot 10-second ECG window ─────────────────────────────────────
DURATION_SEC  = 10
start_sample  = 0
end_sample    = DURATION_SEC * sampling_rate

ecg_window    = ecg_signal[start_sample:end_sample, 0]   # Channel 1 (MLII)
time_axis     = np.arange(len(ecg_window)) / sampling_rate

# ── Find annotation positions within this window ──────────────────
mask          = (beat_samples >= start_sample) & (beat_samples < end_sample)
ann_samples   = beat_samples[mask]
ann_symbols   = np.array(beat_symbols)[mask]

# ── Plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 4))

ax.plot(time_axis, ecg_window, color='#00d4ff', linewidth=0.9, label='ECG Lead MLII')

# Mark beat annotations
colors = {'N': '#00ff88', 'V': '#ff4757', 'A': '#ffa502', 'L': '#eccc68', 'R': '#ff6b81'}
for s, sym in zip(ann_samples, ann_symbols):
    t = (s - start_sample) / sampling_rate
    c = colors.get(sym, '#ffffff')
    ax.axvline(x=t, color=c, alpha=0.5, linewidth=0.8, linestyle='--')
    ax.text(t, ecg_window.max() * 0.95, sym, fontsize=7, color=c, ha='center', fontweight='bold')

# Legend patches
patches = [mpatches.Patch(color=c, label=f'{s} — {BEAT_DESCRIPTIONS.get(s, s)}')
           for s, c in colors.items() if s in ann_symbols]
ax.legend(handles=patches, loc='lower right', fontsize=8,
          facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')

ax.set_title(f'MIT-BIH Record {SAMPLE_RECORD} — Raw ECG Signal (First {DURATION_SEC} seconds)', 
             fontsize=14, color='white', pad=12)
ax.set_xlabel('Time (seconds)', fontsize=11)
ax.set_ylabel('Amplitude (mV)', fontsize=11)
ax.grid(True)
plt.tight_layout()
plt.savefig('./outputs/fig1_raw_ecg_waveform.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig1_raw_ecg_waveform.png')

---
## 📊 Cell 6 — Visualize Both Channels

In [ ]:
# ── Plot both ECG channels side by side ───────────────────────────
channel_names = record.sig_name      # e.g., ['MLII', 'V5']

fig, axes = plt.subplots(2, 1, figsize=(18, 6), sharex=True)
channel_colors = ['#00d4ff', '#ff6b9d']

for i, (ax, color, ch_name) in enumerate(zip(axes, channel_colors, channel_names)):
    ch_window = ecg_signal[start_sample:end_sample, i]
    ax.plot(time_axis, ch_window, color=color, linewidth=0.9)
    ax.set_ylabel(f'{ch_name}\n(mV)', fontsize=10)
    ax.grid(True)
    ax.set_title(f'Channel {i+1}: {ch_name}', fontsize=11, color='white')

axes[-1].set_xlabel('Time (seconds)', fontsize=11)
fig.suptitle(f'MIT-BIH Record {SAMPLE_RECORD} — Dual Channel ECG ({DURATION_SEC}s)', 
             fontsize=14, color='white', y=1.01)
plt.tight_layout()
plt.savefig('./outputs/fig2_dual_channel_ecg.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig2_dual_channel_ecg.png')

---
## 📊 Cell 7 — Beat Type Distribution Bar Chart

In [ ]:
# ── Aggregate beat counts across ALL 48 MIT-BIH records ──────────
# (This takes ~30-60 sec — loads all records)

all_labels = []
records_loaded = 0

for rec_id in MIT_BIH_RECORDS:
    try:
        rec_path = os.path.join(MIT_BIH_PATH, rec_id)
        ann = wfdb.rdann(rec_path, 'atr')
        all_labels.extend(ann.symbol)
        records_loaded += 1
    except Exception:
        pass   # skip if record not downloaded

print(f'✅ Loaded {records_loaded}/{len(MIT_BIH_RECORDS)} records')
print(f'📊 Total annotations collected: {len(all_labels)}')

# ── Count + filter top beat types ────────────────────────────────
label_series  = pd.Series(all_labels)
label_counts_all = label_series.value_counts()

# Keep only valid AAMI beat symbols
VALID_BEATS   = ['N', 'L', 'R', 'A', 'a', 'J', 'S', 'V', 'F', 'e', 'j', 'E', '/', 'f', 'Q']
beat_counts   = label_counts_all[label_counts_all.index.isin(VALID_BEATS)]

# ── Plot ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
bar_colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(beat_counts)))
axes[0].bar(beat_counts.index, beat_counts.values, color=bar_colors, edgecolor='white', linewidth=0.4)
axes[0].set_title('Beat Type Distribution — All MIT-BIH Records', fontsize=12, color='white')
axes[0].set_xlabel('Beat Symbol', fontsize=10)
axes[0].set_ylabel('Count', fontsize=10)
axes[0].grid(axis='y')
for bar, val in zip(axes[0].patches, beat_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', va='bottom', fontsize=8, color='white')

# Pie chart (top 6 categories)
top6 = beat_counts.head(6)
wedge_props = dict(width=0.5, edgecolor='#0f0f0f', linewidth=2)
axes[1].pie(top6.values, labels=top6.index, autopct='%1.1f%%',
            colors=plt.cm.Set2.colors, wedgeprops=wedge_props,
            textprops={'color': 'white', 'fontsize': 11})
axes[1].set_title('Top 6 Beat Types (Proportion)', fontsize=12, color='white')

plt.tight_layout()
plt.savefig('./outputs/fig3_beat_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig3_beat_distribution.png')

---
## 📊 Cell 8 — Build Dataset Statistics Summary Table

In [ ]:
# ── Collect per-record statistics ────────────────────────────────
stats_rows = []

for rec_id in MIT_BIH_RECORDS[:10]:   # preview first 10 records
    try:
        rec_path = os.path.join(MIT_BIH_PATH, rec_id)
        rec = wfdb.rdrecord(rec_path)
        ann = wfdb.rdann(rec_path, 'atr')
        sig = rec.p_signal[:, 0]      # channel 1
        syms = ann.symbol
        unique_syms = list(set(syms))

        stats_rows.append({
            'Record'          : rec_id,
            'Duration (s)'    : round(rec.sig_len / rec.fs, 1),
            'Sampling Rate'   : rec.fs,
            'Total Beats'     : len(syms),
            'Beat Types'      : ', '.join(sorted(set(syms))),
            'Signal Min (mV)' : round(float(sig.min()), 4),
            'Signal Max (mV)' : round(float(sig.max()), 4),
            'Signal Std'      : round(float(sig.std()), 4),
        })
    except Exception as e:
        stats_rows.append({'Record': rec_id, 'Duration (s)': 'Error'})

df_stats = pd.DataFrame(stats_rows)

# ── Display styled table ──────────────────────────────────────────
print('\n📋 MIT-BIH Dataset Statistics — First 10 Records')
print('=' * 90)
print(df_stats.to_string(index=False))
print('=' * 90)

# Save to CSV
df_stats.to_csv('./outputs/mit_bih_stats.csv', index=False)
print('\n✅ Statistics saved → outputs/mit_bih_stats.csv')

---
## 🫀 Cell 9 — Load and Explore PTB Diagnostic ECG Dataset

In [ ]:
# ── PTB Dataset has subfolder structure: patient folders ──────────
# e.g., ptb/patient001/s0010lre

ptb_records = []

if os.path.exists(PTB_PATH):
    for root, dirs, files in os.walk(PTB_PATH):
        for file in files:
            if file.endswith('.hea'):
                rec_name = os.path.splitext(file)[0]
                rec_path = os.path.join(root, rec_name)
                ptb_records.append(rec_path)

    print(f'📋 PTB Dataset: {len(ptb_records)} records found')

    # Load first PTB record
    if ptb_records:
        ptb_rec = wfdb.rdrecord(ptb_records[0])
        ptb_sig = ptb_rec.p_signal

        print('\n📄 Sample PTB Record Info:')
        print(f'   Record Name    : {ptb_rec.record_name}')
        print(f'   Channels       : {ptb_rec.n_sig}  →  {ptb_rec.sig_name}')
        print(f'   Total Samples  : {ptb_rec.sig_len}')
        print(f'   Sampling Rate  : {ptb_rec.fs} Hz')
        print(f'   Duration       : {ptb_rec.sig_len / ptb_rec.fs:.1f} s')
        print(f'   Signal Shape   : {ptb_sig.shape}')

        if ptb_rec.comments:
            print('\n💬 Clinical Comments/Diagnosis:')
            for comment in ptb_rec.comments:
                print(f'   {comment}')
else:
    print('❌ PTB folder not found — verify PTB_PATH')

---
## 📊 Cell 10 — Visualize PTB ECG (12-Lead)

In [ ]:
# ── Plot first 5 seconds of the PTB ECG (4 leads) ─────────────────
if ptb_records and 'ptb_rec' in dir():
    ptb_fs   = ptb_rec.fs
    end_s    = 5 * ptb_fs
    ptb_time = np.arange(end_s) / ptb_fs

    n_leads_to_plot = min(4, ptb_rec.n_sig)
    fig, axes = plt.subplots(n_leads_to_plot, 1, figsize=(18, 2.5 * n_leads_to_plot), sharex=True)
    lead_colors = ['#00d4ff', '#ff6b9d', '#00ff88', '#ffd32a']

    for i in range(n_leads_to_plot):
        lead_sig = ptb_rec.p_signal[:end_s, i]
        axes[i].plot(ptb_time, lead_sig, color=lead_colors[i], linewidth=0.9)
        axes[i].set_ylabel(f'{ptb_rec.sig_name[i]}\n(mV)', fontsize=9)
        axes[i].grid(True)

    axes[-1].set_xlabel('Time (seconds)', fontsize=11)
    fig.suptitle(f'PTB ECG — {ptb_rec.record_name} (First 5 seconds, 4 Leads)', 
                 fontsize=13, color='white')
    plt.tight_layout()
    plt.savefig('./outputs/fig4_ptb_ecg_leads.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print('✅ Figure saved → outputs/fig4_ptb_ecg_leads.png')
else:
    print('⚠️  PTB data not loaded — skipping plot')

---
## 📊 Cell 11 — Dataset Comparison Summary

In [ ]:
# ── Final summary comparison table ───────────────────────────────
summary_data = {
    'Property'              : ['Source', 'Total Records', 'Sampling Rate', 'Channels',
                               'Duration per Record', 'Annotation Type', 'Primary Use'],
    'MIT-BIH Arrhythmia'   : ['PhysioNet', '48 records', '360 Hz', '2 (MLII, V5)',
                               '~30 minutes', 'Beat-level labels', 'Training (Primary)'],
    'PTB Diagnostic ECG'   : ['PhysioNet', '549 records', '1000 Hz', '15 leads',
                               '~2 minutes', 'Diagnosis comments', 'Validation (External)'],
}

df_summary = pd.DataFrame(summary_data).set_index('Property')

print('\n📋 Dataset Comparison Summary')
print('=' * 70)
print(df_summary.to_string())
print('=' * 70)

df_summary.to_csv('./outputs/dataset_comparison_summary.csv')
print('\n✅ Summary saved → outputs/dataset_comparison_summary.csv')
print('\n🎉 Notebook 1 Complete! Dataset loading & exploration done.')
print('   ➡️  Proceed to Notebook 2 — ECG Signal Preprocessing')

---
## ✅ Week 1 — Notebook 1 Summary

| Task | Status |
|------|--------|
| Libraries imported | ✅ Done |
| MIT-BIH dataset loaded | ✅ Done |
| PTB dataset loaded | ✅ Done |
| ECG signal structure inspected | ✅ Done |
| Beat annotations explored | ✅ Done |
| ECG waveforms visualized | ✅ Done |
| Dataset statistics generated | ✅ Done |
| Figures saved | ✅ Done |

**Next Step → Notebook 2: ECG Signal Preprocessing**